# [Sensitivity Analysis] The Naive Baseline (KAN-TabNet) | Step LR | Higgs Boson

**Optimized Reference Configuration:** `n_d=9, n_a=9`, `kan_grid_size=3`, `kan_spline_order=3`, `initial_lr=0.0025`

### Methodological Context: The Control Variables
To demonstrate the necessity of our optimized reference configuration, this sensitivity analysis evaluates the standard, "out-of-the-box" default hyperparameters of the KAN-TabNet architecture. To ensure a fair comparison, we maintain the exact same continuous mathematical geometry (`k=3` cubic splines) and thermodynamic optimization environment (StepLR schedule, `initial_lr=0.0025`) as our optimized reference configuration.

By fixing the optimization environment, we strictly guarantee that any performance variations observed in this notebook are purely the result of the default structural routing capacity ($5 \times 5$) and standard grid resolution ($G=5$).

### The Hypothesis
In this study, we evaluate the naive baseline configuration (`n_d=5, n_a=5`, `kan_grid_size=5`).

The Higgs Boson dataset consists of 28 continuous physical features representing kinematic properties of particle jets. We hypothesize that this default architecture suffers from two simultaneous structural flaws when applied to continuous physics data. First, the narrow $5 \times 5$ routing matrix lacks the attention bandwidth to process all 28 features effectively, creating a severe information bottleneck. Second, the default grid resolution of $G=5$ provides excessive mathematical flexibility, allowing the B-splines to overfit the inherent quantum noise of the dataset rather than capturing the underlying smooth invariant mass curves. This notebook serves to empirically evaluate whether this naive baseline underperforms, thereby validating our strategy of widening the pipeline while aggressively regularizing the grid.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Notes:
# - momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 37 epochs approximates the paper's 20k iterations
#   (based on a batch size of 16384 and ~8.8M training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=37, gamma=0.9),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=5,
    n_a=5,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025),
    use_kan=True,
    kan_grid_size=5,
    kan_spline_order=3,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.6403  | valid_accuracy: 0.67045 |  0:01:50s
epoch 25 | loss: 0.54497 | valid_accuracy: 0.71884 |  0:47:24s
epoch 50 | loss: 0.5356  | valid_accuracy: 0.72639 |  1:32:53s
epoch 75 | loss: 0.52793 | valid_accuracy: 0.73088 |  2:18:34s
epoch 100| loss: 0.52617 | valid_accuracy: 0.73196 |  3:04:27s
epoch 125| loss: 0.52438 | valid_accuracy: 0.73316 |  3:50:48s
epoch 150| loss: 0.52321 | valid_accuracy: 0.7338  |  4:36:41s
epoch 175| loss: 0.52208 | valid_accuracy: 0.73244 |  5:22:11s
epoch 200| loss: 0.52023 | valid_accuracy: 0.73589 |  6:07:57s
epoch 225| loss: 0.51836 | valid_accuracy: 0.737   |  6:53:43s
epoch 250| loss: 0.51671 | valid_accuracy: 0.73657 |  7:39:27s
epoch 275| loss: 0.51504 | valid_accuracy: 0.73875 |  8:25:13s
epoch 300| loss: 0.51196 | valid_accuracy: 0.74118 |  9:10:57s
epoch 325| loss: 0.51125 | valid_accuracy: 0.74185 |  9:56:42s
epoch 350| loss: 0.50899 | valid_accuracy: 0.74339 |  10:42:25s
epoch 375| loss: 0.50891 | valid_accuracy: 0.74293 |  

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 33,662


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.74478


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/05_kan_sensitivity_5x5_grid_5_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/05_kan_sensitivity_5x5_grid_5_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/05_kan_sensitivity_5x5_grid_5_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/05_kan_sensitivity_5x5_grid_5_33k.zip
